# import

In [147]:
import argparse
import os
import copy
import sys

import numpy as np
import torch
from PIL import Image, ImageDraw, ImageFont
from torchvision.ops import box_convert

# Grounding DINO
import GroundingDINO.groundingdino.datasets.transforms as T
from GroundingDINO.groundingdino.models import build_model
from GroundingDINO.groundingdino.util import box_ops
from GroundingDINO.groundingdino.util.slconfig import SLConfig
from GroundingDINO.groundingdino.util.utils import clean_state_dict, get_phrases_from_posmap
from GroundingDINO.groundingdino.util.inference import annotate, load_image, predict

import supervision as sv

# segment anything
base_path = r"C:\\Users\\test\\kwangwoon\\2024_1\\산학연계\\image_searching\segment-anything"

# 경로를 sys.path에 추가
if base_path not in sys.path:
    sys.path.append(base_path)

from segment_anything import build_sam, SamPredictor 
import cv2
import numpy as np
import matplotlib.pyplot as plt
from huggingface_hub import hf_hub_download

# embedding / vector DB
import clip
import psycopg2
from psycopg2.extras import execute_batch

In [2]:
print(torch.__version__)
print(torch.cuda.is_available())

2.3.1+cu121
True


# Grounded SAM

In [3]:
def load_model_hf(repo_id, filename, ckpt_config_filename, device='cpu'):
    cache_config_file = hf_hub_download(repo_id=repo_id, filename=ckpt_config_filename)

    args = SLConfig.fromfile(cache_config_file) 
    model = build_model(args)
    args.device = device

    cache_file = hf_hub_download(repo_id=repo_id, filename=filename)
    checkpoint = torch.load(cache_file, map_location='cpu')
    log = model.load_state_dict(clean_state_dict(checkpoint['model']), strict=False)
    print("Model loaded from {} \n => {}".format(cache_file, log))
    _ = model.eval()
    return model  

In [4]:
ckpt_repo_id = "ShilongLiu/GroundingDINO"
ckpt_filenmae = "groundingdino_swinb_cogcoor.pth"
ckpt_config_filename = "GroundingDINO_SwinB.cfg.py"

In [5]:
groundingdino_model = load_model_hf(ckpt_repo_id, ckpt_filenmae, ckpt_config_filename)

final text_encoder_type: bert-base-uncased
Model loaded from C:\Users\test\.cache\huggingface\hub\models--ShilongLiu--GroundingDINO\snapshots\a94c9b567a2a374598f05c584e96798a170c56fb\groundingdino_swinb_cogcoor.pth 
 => _IncompatibleKeys(missing_keys=[], unexpected_keys=['label_enc.weight', 'bert.embeddings.position_ids'])


In [6]:
DEVICE = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
print(DEVICE)
sam_checkpoint = 'sam_vit_h_4b8939.pth'
sam = build_sam(checkpoint=sam_checkpoint)
sam.to(device=DEVICE)
sam_predictor = SamPredictor(sam)

cuda:0


## 함수 정의

In [136]:
# grounding DINO로 box detection
def detect(image, text_prompt, model, image_source, box_threshold = 0.3, text_threshold = 0.25):
  boxes, logits, phrases = predict(
      model=model, 
      image=image, 
      caption=text_prompt,
      box_threshold=box_threshold,
      text_threshold=text_threshold,
      device=DEVICE
  )
  
  annotated_frame = annotate(image_source=image_source, boxes=boxes, logits=logits, phrases=phrases)
  annotated_frame = annotated_frame[...,::-1] # BGR to RGB 

  return boxes, annotated_frame


# 얻은 박스를 프롬프트로 활용하여 SAM 적용
def segment(image, sam_model, boxes):
  sam_model.set_image(image)
  H, W, _ = image.shape
  boxes_xyxy = box_ops.box_cxcywh_to_xyxy(boxes) * torch.Tensor([W, H, W, H])

  transformed_boxes = sam_model.transform.apply_boxes_torch(boxes_xyxy.to(DEVICE), image.shape[:2])
  masks, _, _ = sam_model.predict_torch(
      point_coords = None,
      point_labels = None,
      boxes = transformed_boxes,
      multimask_output = False,
      )
  return masks.cpu()


# 상품 크기에 맞추어 이미지 crop
def apply_mask_to_image(image, mask):
    # 마스크된 부분만을 원본 이미지에 적용합니다.
    masked_image = image.copy()
    masked_image[mask == 0] = [255, 255, 255]

    masked_indices = np.where(mask != 0)

    # 마스크된 부분의 최소 및 최대 y, x 좌표를 찾습니다.
    min_y, min_x = np.min(masked_indices, axis=1)
    max_y, max_x = np.max(masked_indices, axis=1)

    # 마스크된 부분만을 포함하는 새로운 이미지 생성
    masked_region_only = masked_image[min_y:max_y+1, min_x:max_x+1, :]

    return masked_region_only



## 5가지 이미지만 Grounded SAM 적용 후 결과 확인

In [25]:
image_paths_and_categories = {'assets/피그먼트 스웨트셔츠_SPMWE23C20.jpg': 'sweat/shirts',
                              'assets/에센셜 후드 스웻 셔츠 (MELANGE GREY).jpg': 'hood t-shirts',
                              'assets/[한소희 착용] 414508 여 카고 조거 팬츠_3color.jpg': 'training / jogger pants',
                              'assets/시티보이 빅 오버핏 빅사이즈 쿨 린넨 셔츠 5 COLOR.jpg': 'shirts/blouse',
                              'assets/에센셜 후드 스웻 셔츠 (OATMEAL).jpg': 'hood t-shirts'}

In [28]:
image_paths = list(image_paths_and_categories.keys())
image_source_and_image = {}

for image_path in image_paths:
    image_source, image = load_image(image_path)
    image_source_and_image[image_path] = [image_source, image]

In [32]:
# grounded SAM pipeline
def SAM(prompt, image_source):
    detected_boxes, annotated_frame = detect(image, prompt, image_source=image_source, model=groundingdino_model)
    segmented_frame_masks = segment(image_source, sam_predictor, boxes=detected_boxes)
    masked_region_only = apply_mask_to_image(image_source, segmented_frame_masks[0][0])
    return masked_region_only, annotated_frame

In [108]:
sam_images = []

for item in list(image_paths_and_categories.keys()):
    prompt = image_paths_and_categories[item]
    print(prompt)
    image_source = image_source_and_image[item][0]
    image = image_source_and_image[item][1]
    sam_image, annotated_frame = SAM(prompt, image_source)
    sam_images.append(Image.fromarray(sam_image))

sweat/shirts


SupervisionWarnings: annotate is deprecated: `BoxAnnotator` is deprecated and will be removed in `supervision-0.22.0`. Use `BoundingBoxAnnotator` and `LabelAnnotator` instead


hood t-shirts


SupervisionWarnings: annotate is deprecated: `BoxAnnotator` is deprecated and will be removed in `supervision-0.22.0`. Use `BoundingBoxAnnotator` and `LabelAnnotator` instead


training / jogger pants


SupervisionWarnings: annotate is deprecated: `BoxAnnotator` is deprecated and will be removed in `supervision-0.22.0`. Use `BoundingBoxAnnotator` and `LabelAnnotator` instead


shirts/blouse


SupervisionWarnings: annotate is deprecated: `BoxAnnotator` is deprecated and will be removed in `supervision-0.22.0`. Use `BoundingBoxAnnotator` and `LabelAnnotator` instead


hood t-shirts


SupervisionWarnings: annotate is deprecated: `BoxAnnotator` is deprecated and will be removed in `supervision-0.22.0`. Use `BoundingBoxAnnotator` and `LabelAnnotator` instead


In [158]:
def concatenate_images_horizontally(images):
    # 각 이미지의 너비와 높이를 추출
    widths, heights = zip(*(i.size for i in images))
    
    # 총 너비와 최대 높이 계산
    total_width = sum(widths)
    max_height = max(heights)
    
    # 새 이미지 생성
    new_image = Image.new('RGB', (total_width, max_height))
    
    # 각 이미지를 새 이미지에 붙여넣기
    x_offset = 0
    for img in images:
        new_image.paste(img, (x_offset, 0))
        x_offset += img.width  # 다음 이미지 시작 위치 업데이트
    
    return new_image

# sam_images 리스트에 있는 PIL Image 객체들을 수평으로 합치기
concatenated_image = concatenate_images_horizontally(sam_images)

# 결과 확인
concatenated_image.show()

# CLIP으로 임베딩 후 pgvector에 저장

In [106]:
model, preprocess = clip.load('ViT-B/32', device=DEVICE)

In [137]:
image_features = []

with torch.no_grad():
    for sam_image in sam_images:
        preprocessed_image = preprocess(sam_image).unsqueeze(0).to(DEVICE)
        image_feature = model.encode_image(preprocessed_image)
        image_features.append(image_feature[0])

In [138]:
conn = psycopg2.connect("dbname=imagevector user=test password=5303 host=localhost")

In [139]:
cur = conn.cursor()

In [145]:
# 테이블 초기화. 테스트용
cur.execute("TRUNCATE TABLE items;")
conn.commit()

In [141]:
processed_features = []

for feature in image_features:
    processed_feature = feature.cpu().numpy().tolist()
    processed_features.append(processed_feature)

In [155]:
# 벡터를 데이터베이스에 배치로 삽입
sql = "INSERT INTO items (embedding) VALUES (%s);"
data = [(vector,) for vector in processed_features]  # 각 벡터를 튜플 형태로 포장
execute_batch(cur, sql, data)
conn.commit()

In [156]:
# 데이터베이스에서 모든 레코드 조회
cur.execute("SELECT id, embedding FROM items")
rows = cur.fetchall()

# 조회 결과 출력
for row in rows:
    print("ID:", row[0])
    print("Vector:", row[1])


ID: 1
Vector: [-0.07684326,0.037322998,0.15930176,0.23449707,0.16516113,0.06976318,-0.28857422,0.35668945,0.27807617,-0.25585938,0.033233643,0.14794922,0.07714844,-0.078125,-0.24450684,0.3684082,0.29833984,0.33691406,0.12854004,-0.2878418,-0.7397461,0.17260742,0.08148193,-0.4963379,-0.011833191,-0.052978516,-0.31689453,-0.34594727,0.122924805,-0.018295288,0.09954834,-0.10925293,-0.025924683,-0.14367676,0.10430908,-0.053527832,-0.19665527,0.23803711,-0.018051147,1.2568359,-0.5859375,-0.13684082,-0.13647461,-0.07287598,0.19885254,-1.2314453,0.33935547,-0.044525146,0.40112305,-0.5385742,0.26416016,-0.25976562,0.17785645,0.0647583,-0.2919922,0.21801758,0.10559082,0.17773438,-0.15466309,0.11315918,0.27685547,-0.25463867,0.28442383,0.0362854,-0.20788574,0.3876953,0.29589844,-0.116882324,0.057128906,0.04824829,-0.5019531,0.2836914,-0.16223145,-0.06878662,-0.50634766,-0.19250488,0.5151367,-0.07110596,0.1850586,0.5151367,-0.34472656,-0.83447266,-0.4897461,0.015899658,0.21838379,0.05783081,0.610